# El Ciclo de la Inacción: Agua, Salud y Pobreza en México
**Equipo:** PCABoys (Carlo Kiliano Ferrera, José Julian Pérez, Aldo Sebastián Altamirano)  
**HackODS UNAM 2026 - Instituto de Geofísica**

> *"Limpiar el agua cuesta menos que seguir pagando hospitales: la inacción es más cara."*

### Resumen
Este notebook documenta el proceso analítico para demostrar la relación causal entre tres Objetivos de Desarrollo Sostenible (ODS):
1. **ODS 6 (Agua y Saneamiento):** El origen físico de la toxicidad en cuerpos de agua superficiales.
2. **ODS 3 (Salud y Bienestar):** El impacto biológico reflejado en la morbilidad por enfermedades gastrointestinales.
3. **ODS 1 (Fin de la Pobreza):** La vulnerabilidad de infraestructura y la trampa económica que representan los gastos médicos para familias marginadas.

## 2- Preparar entorno en Colab (nuestro notebook)
En esta primera fase, inicializamos nuestro motor de análisis (Pandas, Plotly, Seaborn y más) para que en todo nuestro notebook podamos trabajar de manera correcta

In [ ]:
# Instalar librerías necesarias
!pip install pyxlsb openpyxl

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import os

# En el caso de usar Google Drive, monta y cambia la ruta
from google.colab import drive, files
drive.mount('/content/drive')

Mounted at /content/drive


###2.1 Carga de archivos y exploración
Para garantizar la reproducibilidad de nuestro proyecto, todos los datos deben  pasar por un proceso de limpieza previo el cual realizaremos en toda la seccion 3 del notebook y provienen de fuentes gubernamentales abiertas:

* **CONAGUA (RENAMECA):** Calidad bacteriológica y química del agua.
* **SSA (Anuarios de Morbilidad):** Incidencia de enfermedades hídricas.
* **CONEVAL:** Medición multidimensional de la pobreza.
* **INEGI (ITER):** Infraestructura básica de vivienda y drenaje.

Procederemos a ver las dimensiones y nombres de columnas de nuestros DataSets Crudos para ver con que nos vamos a enfrentar:

In [ ]:
ruta = '/content/drive/MyDrive/IIMAS/Hackathon/DatosCrudos'
archivos = [f for f in os.listdir(ruta) if f.endswith(('.xlsx', '.xls', '.csv', '.xlsb'))]

print(f"Procesando {len(archivos)} archivos...\n")

for archivo in archivos:
    path_completo = os.path.join(ruta, archivo)
    try:
        if archivo.endswith('.csv'):
            try:
                # Intento 1: Estándar UTF-8
                df_headers = pd.read_csv(path_completo, nrows=0)
            except UnicodeDecodeError:
                # Intento 2: Codificación para español (Latin-1)
                df_headers = pd.read_csv(path_completo, nrows=0, encoding='latin-1')

        elif archivo.endswith('.xlsb'):
            df_headers = pd.read_excel(path_completo, engine='pyxlsb', nrows=0)

        else:
            df_headers = pd.read_excel(path_completo, nrows=0)

        print(f"📄 Archivo: {archivo}")
        print(f"📊 Columnas: {df_headers.columns.tolist()}")
        print("-" * 50)

    except Exception as e:
        print(f"Error persistente al leer {archivo}: {e}")

Procesando 7 archivos...

📄 Archivo: pobreza_grupos_poblacionales_rural_urbano.csv
📊 Columnas: ['clave_entidad', 'entidad_federativa', 'clave_municipio', 'municipio', 'grupo', 'poblacion', 'pobreza_porcentaje', 'carencia_rezago_educativo_porcentaje', 'carencia_servicios_de_salud_porcentaje', 'carencia_seguridad_social_porcentaje', 'carencia_calidad_espacios_vivienda_porcentaje', 'carencia_servicios_basicos_vivienda_porcentaje', 'carencia_alimentacion_nutritiva_calidad_porcentaje', 'ingreso_inferior_a_lpi_porcentaje', 'periodo', 'entidad', 'entidad_federativa_etq']
--------------------------------------------------
📄 Archivo: TODOS LOS MONITOREOS.xlsb
📊 Columnas: ['CLAVE SITIO', 'NOMBRE DEL SITIO', 'CUENCA', 'CLAVE ACUÍFERO', 'ACUÍFERO', 'ORGANISMO CUENCA', 'DIRECCIÓN LOCAL', 'ESTADO', 'MUNICIPIO', 'CUERPO DE AGUA', 'TIPO DE CUERPO DE AGUA', 'SUBTIPO CUERPO AGUA', 'LATITUD', 'LONGITUD']
--------------------------------------------------
📄 Archivo: conjunto_de_datos_iter_00CSV20.csv
📊 Co

In [ ]:
# Morbilidad (enfermedades GI)
df_morb = pd.read_csv('t2017.csv', encoding='latin1')
print(f"Morbilidad shape: {df_morb.shape}")
print("-" * 40)

# ITER (INEGI, población y vivienda)
df_iter = pd.read_csv('conjunto_de_datos_iter_00CSV20.csv', encoding='latin1', low_memory=False)
print(f"ITER shape: {df_iter.shape}")
print("-" * 40)

# Pobreza CONEVAL
df_pobreza = pd.read_csv('pobreza_grupos_poblacionales_rural_urbano.csv', encoding='latin1')
print(f"Pobreza shape: {df_pobreza.shape}")
print("-" * 40)

# Sitios de monitoreo CONAGUA (todos los monitoreos)
try:
    df_conagua = pd.read_excel('TODOS LOS MONITOREOS.xlsb', engine='pyxlsb')
    print(f"TODOS LOS MONITOREOS shape: {df_conagua.shape}")
except Exception as e:
    print(f"Error en TODOS LOS MONITOREOS: {e}")

Morbilidad shape: (4672, 96)
----------------------------------------
ITER shape: (195662, 286)
----------------------------------------
Pobreza shape: (12148, 17)
----------------------------------------
TODOS LOS MONITOREOS shape: (7771, 14)


### 2.2 Descarga csv de archivo EXCEL
Vamos a descargar como archivo csv cada hoja de datos del archivo EXCEL de la CONAGUA donde vienen TODOS LOS MONITOREOS para poder tratarlos por separado:

In [ ]:
# Ruta de nuestro archivo
archivo_excel_CONAGUA = '/content/drive/MyDrive/IIMAS/Hackathon/DatosCrudos/TODOS LOS MONITOREOS.xlsb'

# Abrimos el archivo y listamos las hojas
xls_file = pd.ExcelFile(archivo_excel_CONAGUA, engine='pyxlsb')
nombres_hojas = xls_file.sheet_names

# Imprimimos los nombres para verlos
print("Nombres de las hojas en el archivo:")
for nombre in nombres_hojas:
    print(f"- {nombre}")

Nombres de las hojas en el archivo:
- Sitios
- Resultados
- Simbología


In [ ]:
# Vamos a leer todas las hojas a la vez
todas_las_hojas = pd.read_excel(archivo_excel_CONAGUA, sheet_name=None, engine='pyxlsb')

# Ahora, iteramos sobre cada hoja que se leyó
for nombre_hoja, dataframe_hoja in todas_las_hojas.items():
    # Construimos el nombre del archivo CSV de salida
    # Reemplazamos espacios por '_' para que el nombre del archivo sea más limpio
    nombre_csv = nombre_hoja.replace(' ', '_') + '.csv'

    # Guardamos el DataFrame como un archivo CSV
    dataframe_hoja.to_csv(nombre_csv, index=False, encoding='utf-8-sig')

    print(f"Hoja '{nombre_hoja}' guardada como '{nombre_csv}' con {len(dataframe_hoja)} filas.")

Hoja 'Sitios' guardada como 'Sitios.csv' con 7771 filas.
Hoja 'Resultados' guardada como 'Resultados.csv' con 129235 filas.
Hoja 'Simbología' guardada como 'Simbología.csv' con 463 filas.
